# TruthLine — Run Guide
### Telco-Specific Customer LLM · Track 3 (Fine-Tuning) · AMD Instinct MI300X

This notebook runs the whole project end to end on the AMD ROCm server:
**setup → fine-tune → evaluate → launch the TruthLine console.**

Pipeline: `guardrails → clarity → semantic cache → intent → hybrid RAG → model router → generation → trust gate`,
with MCP enterprise tools (expert routing, ITSM ticketing, live outage feed) and a feedback data flywheel.

> **How serving works:** the app is a Streamlit web server on port `8501`. The AMD
> notebook exposes it through `jupyter-server-proxy`, so you open it at
> `https://<your-instance-id>/proxy/8501/` (build that URL from your browser's
> address bar — replace everything after the instance id with `/proxy/8501/`).

## 1 · Setup — clone/pull the repo and install dependencies
A fresh GPU session is a clean container, so packages must be reinstalled each time.
Code and saved adapters live in the persistent `/workspace/shared` mount.

In [ ]:
# Clone on first use, or pull the latest if it's already there
%cd /workspace/shared
![ -d amd ] && (cd amd && git pull) || git clone https://github.com/Prash8830/amd.git
%cd /workspace/shared/amd

In [ ]:
# Install all dependencies (transformers, peft, langgraph, langchain, chromadb,
# mcp, streamlit, fastapi, rocm-smi tooling, ...). Takes ~5-10 min on a fresh box.
!pip install -r requirements.txt

In [ ]:
# Sanity check: confirm the GPU is visible and the key libraries import
import torch
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
import datasets, transformers, peft, chromadb, streamlit, mcp, langgraph  # noqa
print('all imports OK')

## 2 · Fine-tune the models (LoRA on AMD MI300X)

We fine-tune two models so the **model router** can right-size compute:
- **Qwen3-14B** — the expert lane (proprietary codes, troubleshooting)
- **Qwen2.5-1.5B** — the fast lane (simple FAQs)

Each is LoRA, bf16, rank 32, 12 epochs (~0.9% of params). Training itself is **~60 seconds**;
the rest is one-time model download + load. Adapters save under `./models/`.

> Already fine-tuned in a previous session? You can **skip to step 4** — adapters persist
> in `./models/` (and a backup copy in `/workspace/shared/FINAL_*`).

In [ ]:
# Expert lane — fine-tune Qwen3-14B (default model in config.py)
!python main.py --mode finetune

In [ ]:
# Fast lane — fine-tune Qwen2.5-1.5B (this is what ACTIVATES the model router)
!BASE_MODEL_ID=Qwen/Qwen2.5-1.5B python main.py --mode finetune

In [ ]:
# Confirm both adapters exist
!ls -1 models/

In [ ]:
# (Optional) checkpoint the trained adapters to persistent storage as a backup
import shutil, time, os
tag = time.strftime('FINAL_%m%d_%H%M')
for m in ['telecom-qlora-qwen3-14b', 'telecom-qlora-qwen2.5-1.5b']:
    src = f'models/{m}'
    if os.path.isdir(src):
        shutil.copytree(src, f'/workspace/shared/{tag}_{m}', dirs_exist_ok=True)
print('checkpointed as', tag)

## 3 · Evaluate — measured accuracy, base vs fine-tuned

Runs 18 **held-out, paraphrased** questions through both the base and fine-tuned model
with identical prompts and retrieval. Scored by deterministic fact-matching (no LLM judge).
Prints a per-category accuracy table, token/latency efficiency, and a step-by-step
compliance metric; writes `eval_results.json` (the Model-quality tab reads this).

Expected headline: **base 22% → fine-tuned 94%**, −74% tokens per answer.

In [ ]:
!python evaluate.py

In [ ]:
# (Optional) side-by-side base vs fine-tuned answers for a few queries
!python compare.py

## 4 · Start the MCP enterprise server  ⚠️ run in a TERMINAL, not here

The MCP server powers expert routing, ITSM ticketing, and the live outage feed.
It must be started **before** the app (the app probes it once at startup) and kept running.

**Open a terminal** (File → New → Terminal) and run:
```bash
cd /workspace/shared/amd && python mcp_server/telecom_mcp.py
```
Leave it open — you'll see it log each tool call live (great for the demo).
Expected: `Telecom enterprise MCP server — http://0.0.0.0:8765/sse`.

*(Optional — the app runs fine without it; the sidebar will just show MCP offline.)*

## 5 · Launch the TruthLine console (Streamlit)

Runs in the background so this notebook stays usable. First load takes ~2 min
while the 14B + 1.5B load to GPU.

Set `USE_LANGGRAPH=1` to run the pipeline on the LangGraph StateGraph backend
(same behaviour, graph-structured).

In [ ]:
# Fresh start for a clean demo: clear feedback + tickets, kill any old server
!rm -f feedback/ground_truth.jsonl tickets.jsonl
!pkill -f streamlit; sleep 2

import subprocess, os
env = dict(os.environ, USE_LANGGRAPH='1')   # drop this arg for the classic backend
dash = subprocess.Popen(['python', 'main.py', '--mode', 'ui'], env=env)
print('TruthLine starting on port 8501 — give it ~2 min, then open the proxy URL below.')

In [ ]:
# Print the URL to open. If the auto-detected host is blank, build it manually:
# take your browser address bar, keep everything before '/lab', add '/proxy/8501/'.
import os
host = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '')
print('Open:  <your-jupyter-base-url>/proxy/8501/')
if host:
    print('Likely:', host.rstrip('/') + '/proxy/8501/')

In [ ]:
# Health check — confirm Streamlit is actually serving locally (expect 200)
import subprocess
print(subprocess.run(
    "curl -s -o /dev/null -w 'local 8501: %{http_code}\\n' http://localhost:8501",
    shell=True, capture_output=True, text=True).stdout)

## 6 · Demo script — queries to type in the console
(Also shown in the app sidebar with one-click copy. Click **New conversation** between scenes.)

| # | Scene | Ask |
|---|-------|-----|
| 1 | Hallucination check (toggle *base model* ON in sidebar) | `What does billing code B-204 mean on my invoice?` |
| 2 | Router → fast lane | `When is my payment due?` |
| 2 | Router + live MCP outage feed | `My internet is very slow since yesterday` |
| 3 | Clarity gate (asks, doesn't guess) | `it's not working` |
| 3 | PII masking | `My number is 555-867-5309 and my internet keeps dropping` |
| 4 | Step-by-step troubleshooting (expert lane) | `My HG-2410 LOS light is red` |
| 4 | Conversation memory (ask both in order) | `My router is an HG-2410.` then `the light is red` |
| 5 | Flywheel: 👍 the answer, then re-ask | `What does error code ERR-2077 mean?` → `What does the ERR-2077 error code mean?` |
| 6 | Self-policing → escalation + MCP ticket | `How much would it cost to replace a lost RT-560X router?` |

Then walk the tabs: **Overview → Observability (live rocm-smi) → Governance (review queue + tickets) → Model quality (22%→94% chart)**.

## 7 · (Optional) vLLM serving — model hosted as an API endpoint

By default the expert lane serves in-process. To serve it via **vLLM** instead
(OpenAI-compatible endpoint, paged attention + batching):

```bash
pip install vllm                 # ROCm wheels; if install fails, skip — in-process is the default
bash scripts/serve_vllm.sh       # builds the merged checkpoint if needed, serves on :8200
```
Then relaunch the app with `VLLM_URL` set:
```python
env = dict(os.environ, VLLM_URL='http://localhost:8200/v1')
subprocess.Popen(['python', 'main.py', '--mode', 'ui'], env=env)
```
The generator probes the endpoint and falls back to in-process if it's unreachable.

## 8 · Other entry points (reference)
```bash
python main.py --mode cli        # interactive CLI (no UI)
python main.py --mode api        # FastAPI backend on :8000 (/chat, /health)
python test_pipeline.py          # non-interactive smoke test (4 canned queries)
python compare.py                # base vs fine-tuned, side by side
```
To stop the dashboard: `!pkill -f streamlit`